# Task 5: JSONPlaceholder Users 데이터를 DataFrame으로 변환하기

## 과제 목표
1. `https://jsonplaceholder.typicode.com/users`에서 `requests` 모듈로 JSON 데이터를 불러온다.
2. 데이터를 **pandas DataFrame**으로 저장한다.
3. 컬럼명이 `"address"`인 Series의 내용을 **별도의 DataFrame으로 변환**한다.

## 학습 목표
1. REST API에서 JSON 데이터를 수집하는 방법을 익힌다.
2. JSON의 **중첩(nested) 구조**를 이해한다.
3. 중첩 딕셔너리를 **평탄화(flattening)**하여 DataFrame으로 변환하는 기법을 배운다.
4. `pd.json_normalize()`와 `pd.DataFrame(tolist())`의 차이를 이해한다.

---
## 1단계: 환경 설정

### 왜 이 두 라이브러리인가?

- **`requests`**: HTTP GET 요청을 보내 JSON 데이터를 수신하는 역할
- **`pandas`**: 수신한 JSON 데이터를 표(DataFrame) 형태로 구조화하는 역할

이 두 라이브러리의 조합은 **"웹에서 데이터 수집 → 분석 가능한 형태로 변환"**이라는
데이터 사이언스의 가장 기본적인 워크플로우를 구성합니다.

In [ ]:
import requests
import pandas as pd

print("라이브러리 로딩 완료!")

---
## 2단계: API에서 JSON 데이터 불러오기

### JSONPlaceholder란?

JSONPlaceholder는 **무료 온라인 REST API 테스트 서비스**입니다.
실제 데이터베이스처럼 동작하는 가짜(fake) API를 제공하여, 개발자가 프론트엔드나 학습용 코드를 테스트할 수 있습니다.

### `/users` 엔드포인트의 데이터 구조

이 API는 10명의 가짜 사용자 정보를 JSON 배열로 반환합니다.
각 사용자 객체의 구조:

```json
{
  "id": 1,
  "name": "Leanne Graham",
  "username": "Bret",
  "email": "Sincere@april.biz",
  "address": {          ← 중첩 딕셔너리!
    "street": "Kulas Light",
    "suite": "Apt. 556",
    "city": "Gwenborough",
    "zipcode": "92998-3874",
    "geo": {            ← 2단계 중첩!
      "lat": "-37.3159",
      "lng": "81.1496"
    }
  },
  "phone": "1-770-736-8031 x56442",
  "website": "hildegard.org",
  "company": { ... }    ← 이것도 중첩 딕셔너리
}
```

### `response.json()`의 동작 원리

HTTP 응답의 본문(Body)은 **문자열(string)**입니다. `response.json()`은:
1. 응답 본문의 JSON 문자열을 파싱(parsing)
2. Python의 **리스트(list)** 또는 **딕셔너리(dict)**로 자동 변환

내부적으로 `json.loads(response.text)`와 동일한 동작을 합니다.

In [ ]:
# API 호출
url = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url)

# 응답 상태 확인
print(f"상태 코드: {response.status_code}")  # 200이면 성공
print(f"응답 데이터 타입: {type(response.json())}")
print(f"사용자 수: {len(response.json())}명")

# JSON 데이터를 Python 객체로 변환
users_data = response.json()

# 첫 번째 사용자 데이터 구조 확인
print("\n첫 번째 사용자 데이터:")
import json
print(json.dumps(users_data[0], indent=2))

---
## 3단계: JSON 데이터를 DataFrame으로 변환

### `pd.DataFrame()`에 JSON 리스트를 넣으면 어떻게 되는가?

JSON 배열(리스트 of 딕셔너리)을 `pd.DataFrame()`에 전달하면:
- 각 딕셔너리가 **하나의 행(row)**이 됩니다
- 딕셔너리의 **키(key)**가 **컬럼명(column)**이 됩니다

그러나 **문제가 있습니다**: `address`와 `company` 같은 중첩 딕셔너리는
DataFrame의 셀 안에 **딕셔너리 그대로** 들어갑니다. 즉, 아직 평탄화되지 않은 상태입니다.

```
id | name          | address                           | ...
1  | Leanne Graham | {'street': 'Kulas Light', ...}    | ...
                      ↑ 딕셔너리가 그대로 들어있음!
```

이 상태에서 `address` 컬럼을 별도로 분해하는 것이 다음 단계의 핵심입니다.

In [ ]:
# JSON 리스트 → DataFrame 변환
df_users = pd.DataFrame(users_data)

print(f"DataFrame 크기: {df_users.shape}  (행: {df_users.shape[0]}, 열: {df_users.shape[1]})")
print(f"\n컬럼 목록: {list(df_users.columns)}")
print(f"\naddress 컬럼의 데이터 타입: {type(df_users['address'].iloc[0])}")

# 전체 DataFrame 출력
df_users

---
## 4단계: `address` 컬럼을 DataFrame으로 변환

### 핵심 과제: 중첩 딕셔너리 → 평탄한 DataFrame

`address` 컬럼의 각 셀에는 아래와 같은 딕셔너리가 들어있습니다:
```python
{
  'street': 'Kulas Light',
  'suite': 'Apt. 556',
  'city': 'Gwenborough',
  'zipcode': '92998-3874',
  'geo': {'lat': '-37.3159', 'lng': '81.1496'}  ← 또 중첩!
}
```

### 방법 1: `pd.DataFrame(series.tolist())`

```python
df_address = pd.DataFrame(df_users['address'].tolist())
```

- `.tolist()`: Series의 각 원소를 Python 리스트로 변환 → `[{dict1}, {dict2}, ...]`
- `pd.DataFrame()`: 딕셔너리 리스트를 DataFrame으로 변환
- **장점**: 간단하고 직관적
- **단점**: `geo`가 여전히 딕셔너리로 남음 → 추가 처리 필요

### 방법 2: `pd.json_normalize()` (권장)

```python
df_address = pd.json_normalize(df_users['address'])
```

- **자동으로 모든 중첩을 평탄화**: `geo.lat`, `geo.lng`처럼 점(`.`) 표기법으로 펼침
- **장점**: 2단계 이상의 중첩도 한 번에 처리
- **단점**: 컬럼명에 점(`.`)이 포함됨

### 두 방법의 결과 비교

| 방법 | `geo` 컬럼 | 결과 컬럼 수 |
|------|-----------|:------------:|
| `DataFrame(tolist())` | `{'lat': '-37.3159', 'lng': '81.1496'}` (딕셔너리 그대로) | 5개 |
| `json_normalize()` | `geo.lat`, `geo.lng` (평탄화됨) | 6개 |

과제에서 요구하는 결과(PDF 이미지 참조)는 `street, suite, city, zipcode, geo.lat, geo.lng`이므로
**방법 2(`json_normalize`)가 적합합니다.**

In [ ]:
# 방법 1: pd.DataFrame(tolist()) — 1단계만 평탄화
df_address_v1 = pd.DataFrame(df_users['address'].tolist())
print("[방법 1] pd.DataFrame(tolist())")
print(f"컬럼: {list(df_address_v1.columns)}")
print(f"geo 컬럼 첫 번째 값: {df_address_v1['geo'].iloc[0]}  ← 딕셔너리 그대로!")
print()
df_address_v1.head(3)

In [ ]:
# 방법 2: pd.json_normalize() — 모든 중첩을 평탄화 (권장)
df_address_v2 = pd.json_normalize(df_users['address'])
print("[방법 2] pd.json_normalize()")
print(f"컬럼: {list(df_address_v2.columns)}")
print(f"geo.lat 첫 번째 값: {df_address_v2['geo.lat'].iloc[0]}  ← 평탄화 완료!")
print()
df_address_v2

---
## 5단계: 최종 결과 확인

### `json_normalize()`의 `sep` 파라미터

기본적으로 중첩 키를 `.`으로 연결합니다 (`geo.lat`).
만약 `_`로 연결하고 싶다면:

```python
pd.json_normalize(data, sep='_')  # → geo_lat, geo_lng
```

### 참고: 원본 DataFrame에서 address를 평탄화하여 합치기

실무에서는 `address`를 분리하지 않고 원본 DataFrame 자체를 평탄화하는 경우도 많습니다:

```python
df_flat = pd.json_normalize(users_data)  # 전체 JSON을 한 번에 평탄화
```

이렇게 하면 `address.street`, `address.city`, `company.name` 등
**모든 중첩 필드가 자동으로 펼쳐진** DataFrame이 생성됩니다.

In [ ]:
# 최종 결과: address DataFrame (json_normalize 방식)
df_address = pd.json_normalize(df_users['address'])

print("=" * 60)
print("최종 결과: address 컬럼 → DataFrame 변환")
print("=" * 60)
print(f"크기: {df_address.shape}")
print(f"컬럼: {list(df_address.columns)}")
print()
df_address

In [ ]:
# (참고) 전체 JSON을 한 번에 평탄화하는 방법
df_all_flat = pd.json_normalize(users_data)
print(f"전체 평탄화 컬럼 ({len(df_all_flat.columns)}개):")
for col in df_all_flat.columns:
    print(f"  - {col}")

---
## 핵심 정리

### 이번 과제에서 배운 것

| 개념 | 핵심 내용 |
|------|----------|
| **JSON 중첩 구조** | API 응답은 종종 딕셔너리 안에 딕셔너리가 있는 다층 구조를 가짐 |
| **`response.json()`** | HTTP 응답 본문의 JSON 문자열을 Python 객체(list/dict)로 자동 변환 |
| **`pd.DataFrame(tolist())`** | 1단계 중첩만 해결. 간단하지만 깊은 중첩에는 부족 |
| **`pd.json_normalize()`** | 모든 깊이의 중첩을 자동 평탄화. 점(`.`) 표기법으로 컬럼명 생성 |

### 핵심 코드 요약

```python
# 1. API 호출
response = requests.get(url)
data = response.json()

# 2. DataFrame 변환
df = pd.DataFrame(data)

# 3. 중첩 컬럼 평탄화
df_address = pd.json_normalize(df['address'])
```